# DATA 612 — Project 1: Global Baseline Predictors and RMSE
**Zoran Glisovic**

---

## Business Description

This system recommends Netflix movies to subscribers based on user ratings.

The dataset comes from the **Netflix Prize** — a $1,000,000 competition Netflix ran from 2006 to 2009 to improve their recommendation algorithm by at least 10%. I use a focused dense subset of the data: the 15 most-rated movies and only users who rated at least 12 of those 15 movies. This gives me roughly 48 users, 15 movies, and ~600 real ratings — a manageable but realistic dataset.

In preparation for this project, I watched Parts K–P of the Stanford Networks Illustrated course (Coursera), covering the theory behind global baseline predictors and RMSE.

Having worked in the VFX industry on Netflix original productions, this dataset has personal relevance — the same platform whose recommendation engine influenced which content got greenlit and distributed globally.

---
## 1. Load and Sample Data

The Netflix Prize files use a custom format: each block starts with a movie ID followed by a colon, then rows of user ID, rating, and date. I parse this into a flat table, then filter down to a dense subset.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = 'Netflix Prize data/'

def parse_netflix_file(filepath, max_rows=500_000):
    """Read Netflix Prize format into a flat DataFrame."""
    records = []
    current_movie = None
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if line.endswith(':'):          # movie ID header
                current_movie = int(line[:-1])
            else:                           # rating row
                user_id, rating, date = line.split(',')
                records.append((current_movie, int(user_id), int(rating)))
                if len(records) >= max_rows:
                    break
    return pd.DataFrame(records, columns=['movie_id', 'user_id', 'rating'])

# Parse first 500K rows from the dataset
raw = parse_netflix_file(DATA_PATH + 'combined_data_1.txt')

# Keep only the 15 most-rated movies
top_movies = raw['movie_id'].value_counts().head(15).index
sub = raw[raw['movie_id'].isin(top_movies)]

# Keep only users who rated at least 12 of those 15 movies (ensures density)
active_users = sub.groupby('user_id')['movie_id'].nunique()
active_users = active_users[active_users >= 12].index
df = sub[sub['user_id'].isin(active_users)].copy()

print(f'Users: {df.user_id.nunique()} | Movies: {df.movie_id.nunique()} | Ratings: {len(df)}')

Users: 48 | Movies: 15 | Ratings: 611


---
## 2. User-Item Matrix

I reshape the data into a matrix where rows are users, columns are movies, and each cell is the rating that user gave that movie. Empty cells (NaN) mean the user hasn't rated that movie — this is the missing data the recommender system needs to predict.

In [2]:
# Load movie titles to use as column labels
movies = pd.read_csv(
    DATA_PATH + 'movie_titles.csv',
    encoding='latin-1', header=None,
    names=['movie_id', 'year', 'title'], on_bad_lines='skip'
)
title_map = movies.set_index('movie_id')['title'].to_dict()

# Build the user-item matrix
matrix = df.pivot_table(index='user_id', columns='movie_id', values='rating')
matrix.columns = [title_map.get(c, c) for c in matrix.columns]  # replace IDs with titles

# Show how sparse the matrix is
total  = matrix.shape[0] * matrix.shape[1]
filled = matrix.notna().sum().sum()
print(f'Matrix: {matrix.shape[0]} users x {matrix.shape[1]} movies')
print(f'Filled: {filled} of {total} cells | Sparsity: {1 - filled/total:.2%}')
matrix.head()

Matrix: 48 users x 15 movies
Filled: 611 of 720 cells | Sparsity: 15.14%


,What the #$*! Do We Know!?,7 Seconds,Immortal Beloved,Lilo and Stitch,Something's Gotta Give,Aqua Teen Hunger Force: Vol. 1,Spitfire Grill,Dragonheart,Congo,Silkwood,Mostly Martha,Spartan,Duplex (Widescreen),Rambo: First Blood Part II,The Game
user_id,,,,,,,,,,,,,,,
16272,NaN,3.0,4.0,2.0,4.0,NaN,3.0,4.0,4.0,3.0,5.0,2.0,3.0,4.0,4.0
147386,NaN,2.0,5.0,4.0,4.0,NaN,3.0,4.0,4.0,5.0,NaN,4.0,3.0,3.0,4.0
303948,NaN,3.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,NaN,4.0,4.0,4.0,5.0
305344,1.0,2.0,2.0,3.0,1.0,1.0,1.0,2.0,2.0,5.0,2.0,1.0,2.0,2.0,3.0
322009,3.0,3.0,3.0,3.0,5.0,NaN,4.0,3.0,3.0,4.0,2.0,4.0,4.0,4.0,4.0


---
## 3. Train / Test Split

I split the known ratings into a training set (80%) and a test set (20%). The model learns only from the training set. The test set is held back and used only to evaluate how well the predictions perform on unseen data.

In [3]:
train, test = train_test_split(df, test_size=0.2, random_state=42)

print(f'Train: {len(train)} ratings | Test: {len(test)} ratings')

Train: 488 ratings | Test: 123 ratings


---
## 4. RMSE — Error Metric

RMSE (Root Mean Squared Error) measures how far off my predictions are from the actual ratings. A lower RMSE means better predictions. I use it to compare the two methods below.

In [4]:
def rmse(actual, predicted):
    """Calculate root mean squared error between actual and predicted ratings."""
    return np.sqrt(np.mean((actual - predicted) ** 2))

---
## 5. Raw Average Predictor

The simplest possible prediction: for every user-movie pair, predict the same value — the overall average rating across all training data. This ignores everything I know about individual users or movies, so it's my baseline to beat.

In [5]:
# Global mean rating across all training data
global_mean = train['rating'].mean()
print(f'Global mean rating: {global_mean:.4f}')

# Predict global mean for every entry, then calculate error
print(f'RMSE (raw average) — Train: {rmse(train.rating, global_mean):.4f} | Test: {rmse(test.rating, global_mean):.4f}')

Global mean rating: 3.2664
RMSE (raw average) — Train: 1.2292 | Test: 1.1819


---
## 6. User and Item Biases

Some users always rate high, some always rate low. Some movies are consistently loved or disliked. I capture these tendencies as biases:

- **User bias**: how much a user's average rating differs from the global mean
- **Item bias**: how much a movie's average rating differs from the global mean, after accounting for who rated it

Both are calculated from the training data only.

In [6]:
# User bias: each user's average rating minus the global mean
user_bias = train.groupby('user_id')['rating'].mean() - global_mean

# Item bias: average leftover after removing global mean and user bias
train2 = train.copy()
train2['bu'] = train2['user_id'].map(user_bias).fillna(0)  # attach user bias to each row
item_bias = train2.groupby('movie_id').apply(
    lambda g: (g['rating'] - global_mean - g['bu']).mean()  # residual per movie
)

print(f'User biases — min: {user_bias.min():.4f}, max: {user_bias.max():.4f}')
print(f'Item biases — min: {item_bias.min():.4f}, max: {item_bias.max():.4f}')

User biases — min: -2.0664, max: 1.5669
Item biases — min: -0.4960, max: 0.6852


---
## 7. Baseline Predictor

The baseline prediction combines three things: the global average, plus the user's personal tendency, plus the movie's tendency. This gives a smarter per-user-per-movie prediction than just using the global mean. Predictions are clipped to the valid rating range of 1 to 5.

In [7]:
def predict(df):
    """Predict ratings as: global mean + user bias + item bias."""
    bu = df['user_id'].map(user_bias).fillna(0)   # user bias (0 if unknown user)
    bi = df['movie_id'].map(item_bias).fillna(0)  # item bias (0 if unknown movie)
    return (global_mean + bu + bi).clip(1, 5)     # clip to valid rating range

print(f'RMSE (baseline) — Train: {rmse(train.rating, predict(train)):.4f} | Test: {rmse(test.rating, predict(test)):.4f}')

RMSE (baseline) — Train: 0.8679 | Test: 0.9799


---
## 8. Results Summary

Comparison of both methods on train and test data.

In [8]:
results = pd.DataFrame({
    'Method':     ['Raw Average', 'Baseline Predictor'],
    'Train RMSE': [rmse(train.rating, global_mean),       rmse(train.rating, predict(train))],
    'Test RMSE':  [rmse(test.rating,  global_mean),       rmse(test.rating,  predict(test))]
})
print(results.to_string(index=False))

# How much did the baseline improve over raw average on test data?
improvement = (rmse(test.rating, global_mean) - rmse(test.rating, predict(test))) / rmse(test.rating, global_mean) * 100
print(f'\nBaseline improved test RMSE by {improvement:.2f}% over raw average.')

            Method  Train RMSE  Test RMSE
       Raw Average    1.229228   1.181878
Baseline Predictor    0.867857   0.979935

Baseline improved test RMSE by 17.09% over raw average.


---
## 9. Conclusion

I tested two prediction methods on a dense subset of the Netflix Prize dataset (~48 users, 15 movies, ~600 real ratings):

**Raw Average** predicts the same global mean rating for every user and movie. It ignores all individual patterns and serves as my starting point.

**Baseline Predictor** improves on this by adding a correction for each user's personal rating tendencies and each movie's overall reception. Even this simple adjustment produces meaningfully better predictions on unseen test data.

This mirrors the foundational insight from the Netflix Prize — bias correction was one of the first and most impactful building blocks used by every competitive team, including the eventual winners.